# Week 7, The AI as a Reader, at Scale  (completion problem, **skeleton** version)

**What you'll do today.** Use a big AI model (Gemini) to *judge* every item in a slice of your
corpus, the way a human coder would, but in minutes.

> **Don't lose your work.** Opened from GitHub, this notebook is read-only: **File → Save a copy in Drive** before editing, and save durable outputs to your Drive project folder — Colab's own disk is wiped when the runtime ends. Course notebooks get updates during the term; to pick them up, open the notebook fresh from GitHub (or `git pull` if you cloned the repo). Updates never touch your saved copy.


In [ ]:
# If an import fails: re-run the install cell above; if it persists see ../kits/common-errors-cheatsheet.md
# (standalone copy: https://github.com/lucianli123/culture-as-data-2026/blob/main/kits/common-errors-cheatsheet.md)
# --- Make your work survive a Colab reset -------------------------------------
# Colab wipes the runtime when it disconnects or idles out. Mount your Google Drive
# and keep everything in ONE project folder, so your corpus, embeddings, labels, and
# charts are still there next week. (Outside Colab - e.g. the offline test harness -
# this falls back to a local folder so the notebook still runs.)
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/culture-as-data"
except Exception:
    PROJECT_DIR = os.path.abspath("./culture-as-data-project")
os.makedirs(PROJECT_DIR, exist_ok=True)
print("Project folder:", PROJECT_DIR)

In [ ]:
# Fetch the pinned package lists if they aren't beside the notebook (this happens
# whenever you open a single notebook straight from GitHub in Colab).
import os, urllib.request
_RAW = "https://raw.githubusercontent.com/lucianli123/culture-as-data-2026/main/notebooks/"
for _f in ("requirements.txt", "constraints.txt"):
    if not os.path.exists(_f):
        urllib.request.urlretrieve(_RAW + _f, _f)

# First, install the few packages Colab doesn't already ship. If you opened this
# notebook with the whole repo, the line below uses our pinned versions:
%pip install -q -r requirements.txt -c constraints.txt

# Opened this notebook on its own, without the repo files? Comment the line above
# and use this explicit pinned install instead:
# %pip install -q google-generativeai pandas

In [ ]:
import os, json
import pandas as pd

# Read the key from a Colab secret / environment variable, NEVER hard-code it.
API_KEY = os.environ.get("GEMINI_API_KEY")
try:
    from google.colab import userdata  # only exists in Colab
    API_KEY = API_KEY or userdata.get("GEMINI_API_KEY")
except Exception:
    pass
LIVE = bool(API_KEY)
print("Live Gemini calls:", LIVE, "(False = using the recorded sample response)")

## Your corpus slice and your codebook (the prompt)

Five real sonnet openings, labeled by dominant theme (`love` / `time` / `beauty`). The
prompt is the codebook: the categories and their boundaries are your analytical choices,
written in prose.

In [ ]:
items = [
    "Shall I compare thee to a summer's day? Thou art more lovely and more temperate",
    "My mistress' eyes are nothing like the sun; coral is far more red than her lips' red",
    "Let me not to the marriage of true minds admit impediments; love is not love which alters",
    "That time of year thou mayst in me behold, when yellow leaves, or none, or few, do hang",
    "When to the sessions of sweet silent thought I summon up remembrance of things past",
]
PROMPT = """You are labeling opening lines of Shakespeare's sonnets by dominant theme.
Return ONLY JSON: a list of objects with keys "label" and "confidence".
"label" is one of: love, time, beauty.
"confidence" is a number from 0 to 1 (how sure you are).
Label each item in order. Items:
{items}"""

### Or better: your own corpus slice

If Week 4's `data.csv` is in your Drive project folder and you have an API key, the cell below
swaps in a slice of *your* corpus. **Rewrite `PROMPT` above first**, it is your codebook, and
the film labels won't fit your data. (Without a key we keep the five demo items, so the recorded
response still lines up.)

In [ ]:
import os
_corpus_file = os.path.join(PROJECT_DIR, "data.csv")
if not os.path.exists(_corpus_file):
    print("no data.csv in your project folder, using the 5 demo items (that's fine)")
elif not LIVE:
    print("no API key, keeping the demo items so the recorded response still matches")
else:
    import pandas as pd
    _df = pd.read_csv(_corpus_file)
    _col = next((c for c in _df.columns if _df[c].dtype == object), None)  # or set _col = "your_column"
    items = _df[_col].dropna().astype(str).head(20).tolist()
    print(f"annotating {len(items)} items from YOUR corpus (text column {_col!r})")

## The recorded sample response (the cassette)

So the notebook runs without a key or network, we commit one recorded response.

In [ ]:
CASSETTE = json.dumps([
    {"label": "love",   "confidence": 0.60},
    {"label": "love",   "confidence": 0.85},
    {"label": "love",   "confidence": 0.95},
    {"label": "time",   "confidence": 0.90},
    {"label": "time",   "confidence": 0.70},
])

## Run the annotator

In [ ]:
# TODO (skeleton): write annotate(items) that builds the prompt, calls Gemini when LIVE
# (model "gemini-2.5-flash") or uses CASSETTE otherwise, strips any code fences, and
# json.loads the result into a list of {"label","confidence"} dicts. Then build a DataFrame
# with the items and print it.

## Confidence. Sort, trust, and hand-check

Trust the labels the model is sure of; pull the **unsure** ones for hand-checking.

In [ ]:
# TODO: sort the DataFrame by confidence ascending; the lowest-confidence rows are the ones
# to check by hand. Print them.

## Accuracy check against a small gold set

You labeled a few by hand.

In [ ]:
# TODO: make a small list `gold` of your own hand labels for the same items, compare to the
# model's labels, and print the disagreements.

## Demo: change the question, change the reader

Watch this first. Same five items, one different question in the codebook, and the machine
becomes a different reader: instead of positive/negative/mixed, ask whether each judgment is
about the film's *craft* or the viewer's *feeling*. The labels aren't sitting in the text
waiting to be found; they come from the question you asked.

In [ ]:
import json

PROMPT2 = """You are labeling short opinions about films.
Return ONLY JSON: a list of objects with keys "label" and "confidence".
"label" is one of: craft, feeling  (is the judgment mostly about the film's craft, or the viewer's feeling?)
"confidence" is a number from 0 to 1 (how sure you are).
Label each item in order. Items:
{items}"""

# recorded response so this demo also runs with no key / offline
CASSETTE2 = json.dumps([
    {"label": "nature",   "confidence": 0.92},
    {"label": "nature",   "confidence": 0.88},
    {"label": "abstract", "confidence": 0.85},
    {"label": "nature",   "confidence": 0.90},
    {"label": "abstract", "confidence": 0.75},
])

def annotate_with(prompt_tmpl, some_items, cassette):
    prompt = prompt_tmpl.format(items="\n".join(f"- {x}" for x in some_items))
    if LIVE:
        import google.generativeai as genai
        genai.configure(api_key=API_KEY)
        raw = genai.GenerativeModel("gemini-2.5-flash").generate_content(prompt).text
    else:
        raw = cassette
    raw = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    return json.loads(raw)

import pandas as pd
df2 = pd.DataFrame(annotate_with(PROMPT2, items, CASSETTE2))
df2["text"] = items
print(df2.to_string(index=False))

## Playground: write a line and try to fool it

Write one line of your own — pastiche Shakespeare, a lyric, anything — and see what the
codebook does with it. Try to write one it gets *wrong*; the confidence is the tell.
(Without a key this cell replays a stand-in answer, so a live judgment needs the key.)

In [ ]:
your_line = "edit me: write one line of your own, then run this cell"

YOUR_CASSETTE = json.dumps([{"label": "love", "confidence": 0.50}])  # offline stand-in only
result = annotate_with(PROMPT, [your_line], YOUR_CASSETTE)[0]
if not LIVE:
    print("(no API key: recorded stand-in, not a real judgment)")
print(f"label: {result['label']}   confidence: {result['confidence']}")

## You read your whole corpus at scale

You borrowed a powerful, opaque reader, made the prompt your codebook, used confidence to decide
what to trust, and checked it against your own judgment.

**Sketch:** show one label the AI got confidently wrong and one it nailed; for each, say how you
knew.